In [ ]:
import warnings
warnings.filterwarnings(action = "ignore")
import numpy as np
import pandas as pd
import ab_sim as sim_mod
from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze
from SALib.sample import saltelli
from SALib.analyze import sobol
from multiprocessing import Pool
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import pickle
import random

In [ ]:
with open("initial_inputs_all.pkl", "rb") as input1:
    initial_inputs = pickle.load(input1)

initial_inputs

In [ ]:
with open("case_inputs_all.pkl", "rb") as input2:
    farm1 = pickle.load(input2)
farm1 = farm1['Farm1']
farm1.keys()

In [ ]:
len(farm1['burn_in_farm'])

In [ ]:
# burn_in_state = dict(random.sample(list(farm1['burn_in_farm'].items()), 500))
burn_in_state = farm1['burn_in_farm']

# Morris sensitivity

In [ ]:
def model_wrapper(params):
    T, R, I, IC, HS = params
    I = int(round(I))
    IC = int(round(IC))
    HS = int(round(HS))
    sim_results = sim_mod.run_main_simulation(burn_in_farm_state = dict(random.sample(list(farm1['burn_in_farm'].items()), HS)), 
                                      merge_dwell_df = sim_mod.merge_dwell, 
                                      ndays = 100, 
                                      burn_in_days = 3650, 
                                      initial_infected = IC, 
                                      transmission_rate = T, 
                                      recovery_rate = R, 
                                      incubation_period = I,
                                      pens_to_mask=['Pen1', 'Pen2', 'Pen3', 'Pen4'],
                                      num_simulations=1)
    history = sim_results['seir_history']
    history['id'] = history['id'].astype(int)
    history['day'] = history['day'].astype(int) 
    history['alive'] = history['alive'].astype(str)  # Now string: 'Alive' or 'Dead'
    history['state'] = history['state'].astype(str)

    # Calculate infected per day (proportion of total individuals per day)
    daily_totals = history.groupby('day').size()
    infected_counts = history[history['state'] == 'Infectious'].groupby('day').size()

    # Align indices and handle missing days
    infected_per_day = infected_counts.reindex(daily_totals.index, fill_value=0) / daily_totals
    peak_infectious = infected_per_day.max() if not infected_per_day.empty and not infected_per_day.isna().all() else 0

    # Calculate total infectious and exposed (proportion of unique individuals)
    total_infectious = len(history[history['state'] == 'Infectious']['id'].unique()) / len(history['id'].unique())
    total_exposed = len(history[history['state'] == 'Exposed']['id'].unique()) / len(history['id'].unique())

    # Calculate final recovered proportion
    max_day = history['day'].max()
    final_state = history[history['day'] == max_day]
    final_recovered = (final_state['state'] == 'Recovered').sum() / len(history['id'].unique()) if not final_state.empty else 0

    outcomes = [float(peak_infectious),
                float(total_infectious),
                float(total_exposed),
                float(final_recovered)]
    return outcomes

In [ ]:
problem = {
    'num_vars': 5,
    'names': ['T', 'R', 'I', 'IC', 'HS'],
    'bounds': [[0.01, 0.99],   # T: e.g., low-high transmission
               [0.01, 0.99],  # R: recovery rate
               [1, 20],     # I: incubation period
               [0, 50],
              [100, len(farm1['burn_in_farm'])]],      # IC: some initial infected cows
    'dists': ['unif', 'unif', 'unif', 'unif', 'unif']  # Uniform; change to 'norm', 'triang', etc. if needed
}

In [ ]:
# Generate Morris samples (r = number of trajectories; more = better accuracy)
r = 20  # Start with 20; total evals = r * (p + 1) = 20 * 5 = 100
morris_samples = morris_sample.sample(problem, N=r, num_levels=6, optimal_trajectories=None)
morris_outputs_list = [model_wrapper(params) for params in morris_samples]
morris_outputs = np.array(morris_outputs_list, dtype=np.float64)  # Force float64 array
print("Morris outputs dtype:", morris_outputs.dtype)  # Debug: should be float64
outcome_names = ['peak_Inf', 'total_Inf', 'total_Ex', 'final_Rec']

In [ ]:
morris_results = []
for i, name in enumerate(outcome_names):
    Y = morris_outputs[:, i].astype(np.float64)
    Si = morris_analyze.analyze(problem, morris_samples, Y,
                                num_levels=6, print_to_console=False)
    
    df = pd.DataFrame({
        'outcome': name,
        'Parameter': problem['names'],
        'Abs_Avg_effect': Si['mu_star'],
        'SD': Si['sigma'],
        'Avg_effect': Si['mu']
    })
    morris_results.append(df)
    print(f"\nMorris Results for {name}:")
    print(df.sort_values('Abs_Avg_effect', ascending=False))
    
    plt.figure(figsize=(8, 4))
    sns.barplot(x='Abs_Avg_effect', y='Parameter', data=df.sort_values('Abs_Avg_effect'))
    plt.title(f'Morris Sensitivity for {name}')
    plt.xlabel('Importance (Abs_Avg_effect)')
    plt.show()

In [ ]:
morris_results

In [ ]:
morris_results = pd.concat(morris_results, ignore_index = True)
morris_results['low_ci'] = morris_results['Avg_effect'] - 2 *morris_results['SD']
morris_results['high_ci'] = morris_results['Avg_effect'] + 2 *morris_results['SD']


morris_results = morris_results.groupby('Parameter').apply(
    lambda x: x.sort_values('Avg_effect', ascending=False)).reset_index(drop=True)

morris_results

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=morris_results,
    x='Parameter',
    y='Avg_effect',
    hue='outcome',
    errorbar=None,
#     palette = "Paired",
    alpha=0
)

for i, patch in enumerate(ax.patches):
    # Get the bar's x, y, and height
    x = patch.get_x() + patch.get_width()/2
    height = patch.get_height()

    # Get matching row
    row = morris_results.iloc[i]
    low = row['low_ci']
    high = row['high_ci']
    
    # Draw vertical line for CI
    ax.plot([x, x], [low, high], color='black', linewidth=0.75, zorder=1)
    # Add small horizontal caps
    ax.plot([x - 0.05, x + 0.05], [low, low], color='black', linewidth=1.2, zorder=1)
    ax.plot([x - 0.05, x + 0.05], [high, high], color='black', linewidth=1.2, zorder=1)
sns.barplot(data=morris_results, 
            x='Parameter', 
            y='Avg_effect', 
            hue='outcome',
            edgecolor='black',
            errorbar=None, 
#             palette = "Paired",
            ax=ax, 
            zorder = 2)    
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title='Outcome')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8, zorder=0)
plt.title("Morris Sensitivity - Average Effect by Parameter", fontsize=14)
plt.ylabel("Avg. effect (% change in outcome)")
plt.xlabel("Parameter")
sns.despine()
plt.tight_layout()
plt.show()

# Sobol sensitivity

In [ ]:
# Generate Saltelli samples for Sobol
N = 256  
sobol_samples = saltelli.sample(problem, N, calc_second_order=True)

sobol_outputs_list = [model_wrapper(params) for params in sobol_samples]
sobol_outputs = np.array(sobol_outputs_list, dtype=np.float64)  # Force float64

In [ ]:
# Analyze Sobol for each outcome
sobol_results = []
S2_results = []
for i, name in enumerate(outcome_names):
    Y = sobol_outputs[:, i].astype(np.float64)
    print(f"Y dtype for Sobol {name}:", Y.dtype)  # Debug
    Si = sobol.analyze(problem, Y, calc_second_order=True, print_to_console=False)
    
    params = problem['names']
    S1 = Si['S1']
    ST = Si['ST']
    S2 = Si['S2']
    
    df_first = pd.DataFrame({
        'Parameter': params,
        'First_Order': S1,
        'Total_Order': ST,
        'Outcome': name
    })

    print(f"\nSobol Results for {name}:")
    print(df_first)
    
#     df_plot = df_first.melt(id_vars='Parameter', var_name='Index', value_name='Value')
#     plt.figure(figsize=(8, 4))
#     sns.barplot(x='Parameter', y='Value', hue='Index', data=df_plot)
#     plt.title(f'Sobol Indices for {name}')
#     plt.ylabel('Sensitivity Index')
#     plt.show()
    sobol_results.append(df_first)
    
    
    if S2 is not None:
        print("Second-order interactions:")
        for j in range(len(params)):
            for k in range(j+1, len(params)):
                print(f"{params[j]}-{params[k]}: {S2[j, k]}")
                s2_result = {'Outcome': name, 'S2': S2[j, k], 'Parameter1':params[j], 'Parameter2':params[k]}
                S2_results.append(s2_result)

In [ ]:
sobol_results

In [ ]:
S2_results = pd.DataFrame(S2_results)
S2_results

In [ ]:
sobol_results = pd.concat(sobol_results, ignore_index = True)
sobol_results

In [ ]:
from matplotlib.patches import Patch
df_long = sobol_results.melt(
    id_vars=['Parameter', 'Outcome'],
    value_vars=['First_Order', 'Total_Order'],
    var_name='Order_Type',
    value_name='Contribution'
)

# Unique levels
parameters = df_long['Parameter'].unique()
outcomes = df_long['Outcome'].unique()
order_types = ['First_Order', 'Total_Order']

# Color mapping by Outcome
palette = sns.color_palette("Set2", n_colors=len(outcomes))
color_map = {o: palette[i] for i, o in enumerate(outcomes)}

plt.figure(figsize=(10, 6))
ax = plt.gca()

bar_width = 0.6 / len(outcomes)  # narrower for multiple outcomes per parameter

# Loop over parameters and outcomes
for i, param in enumerate(parameters):
    for j, outcome in enumerate(outcomes):
        # Filter subset
        subset = df_long[(df_long['Parameter'] == param) &
                         (df_long['Outcome'] == outcome)]

        total_order = subset.loc[subset['Order_Type'] == 'Total_Order', 'Contribution'].values[0]
        first_order = subset.loc[subset['Order_Type'] == 'First_Order', 'Contribution'].values[0]

        # Position offset for outcome within each parameter group
        x = i - 0.3 + j * bar_width

        # Draw stacked bars
        ax.bar(x, first_order, width=bar_width, color=color_map[outcome],
               edgecolor='black', alpha=0.8, label=f"{outcome} (Main)" if i == 0 else "")
        ax.bar(x, total_order, width=bar_width, color=color_map[outcome],
               edgecolor='black', hatch='...', alpha=0.25, label=f"{outcome} (Total)" if i == 0 else "")

# Formatting
ax.set_xticks(np.arange(len(parameters)))
ax.set_xticklabels(parameters)
ax.set_ylabel("Contribution to Model Variance")
ax.set_xlabel("Parameter")
ax.set_title("Sobol Sensitivity", fontsize=14)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
sns.despine()

# Legend
handles_outcome = [Patch(facecolor=color_map[o], edgecolor='black', label=f"{o} (Main effect)") for o in outcomes]
handles_first = [Patch(facecolor='white', edgecolor='black', hatch='//', label="Total_Order")]
ax.legend(handles=handles_outcome + handles_first,
          title='Outcome + Order Type', loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
df_total = sobol_results.melt(
    id_vars=['Parameter', 'Outcome'],
    value_vars=['Total_Order'],
    var_name='Order_Type',
    value_name='Contribution'
)

# Pivot to Outcome × Parameter matrix
df_pivot = df_total.pivot_table(
    index='Outcome',
    columns='Parameter',
    values='Contribution',
    aggfunc='first'
).fillna(0)

# 🔹 Normalize so each outcome sums to 1
df_pivot = df_pivot.div(df_pivot.sum(axis=1), axis=0)

# Optional: sort parameters by mean importance (across outcomes)
param_order = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot[param_order]

# Colors per parameter
palette = sns.color_palette("viridis", n_colors=len(df_pivot.columns))
color_map = {param: palette[i] for i, param in enumerate(df_pivot.columns)}

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(df_pivot))
for param in df_pivot.columns:
    ax.bar(df_pivot.index, df_pivot[param],
           bottom=bottom,
           color=color_map[param],
           edgecolor='black',
           label=param)
    bottom += df_pivot[param]

# Formatting
ax.set_ylabel("Normalized Total Sobol Contribution (sum = 1)")
ax.set_xlabel("Outcome")
ax.set_title("Normalized Total Order Sobol Contributions by Outcome", fontsize=14)
ax.set_ylim(0, 1.05)
sns.despine()

# Legend outside plot
ax.legend(title="Parameter", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:

# Set global font sizes
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

# Get unique outcomes
outcomes = S2_results["Outcome"].unique()
n_outcomes = len(outcomes)

# Create subplots
fig, axes = plt.subplots(1, n_outcomes, figsize=(7 * n_outcomes, 7), constrained_layout=True)
if n_outcomes == 1:
    axes = [axes]

# Get unique parameters
parameters = pd.unique(S2_results[["Parameter1", "Parameter2"]].values.ravel())

for ax, outcome in zip(axes, outcomes):
    # Filter data for the current outcome
    sub_data = S2_results[S2_results["Outcome"] == outcome]
    
    # Create a complete interaction matrix
    matrix = pd.DataFrame(index=parameters, columns=parameters, dtype=float)
    
    # Fill the matrix with S2 values
    for _, row in sub_data.iterrows():
        param1, param2, s2 = row["Parameter1"], row["Parameter2"], row["S2"]
        matrix.at[param1, param2] = s2
        matrix.at[param2, param1] = s2  # Ensure symmetry
    
    # Fill NaN values with 0 for missing interactions
#     matrix = matrix.fillna(0)
    
    # Normalize S2 values for color scale
    max_abs_value = matrix.abs().max().max()
    normalized_data = matrix / max_abs_value if max_abs_value > 0 else matrix
    
    # Create triangular mask and apply to data
    mask = np.tril(np.ones_like(matrix, dtype=bool))
    masked_data = normalized_data.copy()
    masked_data[~mask] = np.nan  # Set upper triangle (excluding diagonal) to NaN
    
    # Convert to numpy array for pcolormesh
    data = masked_data.values
#     print(f"Annotation data for {outcome}:\n{normalized_data.round(2)}")  # Debug print
    
    # Create heatmap using pcolormesh
    x = np.arange(len(parameters) + 1)
    y = np.arange(len(parameters) + 1)
    X, Y = np.meshgrid(x, y)
    pcm = ax.pcolormesh(X, Y, data, cmap="RdBu", vmin=-1, vmax=1)
    
    # Add annotations for the lower triangle
    for i in range(len(parameters)):
        for j in range(len(parameters)):
            if mask[i, j]:  # Only annotate lower triangle
                value = normalized_data.iloc[i, j].round(2)
                ax.text(j + 0.5, i + 0.5, f"{value:.2f}", 
                        ha="center", va="center", 
                        color="black", fontsize=12, weight="bold")
    
    # Add colorbar
    plt.colorbar(pcm, ax=ax, label="Normalized S₂", shrink=0.8)
    
    # Set ticks and labels
    ax.set_xticks(np.arange(len(parameters)) + 0.5)
    ax.set_yticks(np.arange(len(parameters)) + 0.5)
    ax.set_xticklabels(parameters, rotation=45)
    ax.set_yticklabels(parameters)
    
    ax.set_title(outcome, fontsize=20, pad=20)
    ax.set_xlabel("Parameter 2", fontsize=16)
    ax.set_ylabel("Parameter 1", fontsize=16)

# Add overall title
fig.suptitle("Sobol S₂ Interactions", fontsize=22, y=1.08)
plt.show()